
---

# 📘 LeetCode 1919: Leetcodify Similar Friends

**Level:** Hard  

---

## ❓ Question

We want to find **similar friends** in Leetcodify.  

Two users `x` and `y` are considered similar friends if:  
1. They are friends.  
2. They listened to the same **three or more different songs** on the **same day**.  

Return all similar pairs of friends in the same format as the input (i.e., always `user1_id < user2_id`).  

---

## 📊 Sample Data

### Listens Table

| user_id | song_id | day        |
|---------|---------|------------|
| 1       | 10      | 2021-03-15 |
| 1       | 11      | 2021-03-15 |
| 1       | 12      | 2021-03-15 |
| 2       | 10      | 2021-03-15 |
| 2       | 11      | 2021-03-15 |
| 2       | 12      | 2021-03-15 |
| 3       | 10      | 2021-03-15 |
| 3       | 11      | 2021-03-15 |
| 3       | 12      | 2021-03-15 |
| 4       | 10      | 2021-03-15 |
| 4       | 11      | 2021-03-15 |
| 4       | 13      | 2021-03-15 |
| 5       | 10      | 2021-03-16 |
| 5       | 11      | 2021-03-16 |
| 5       | 12      | 2021-03-16 |

### Friendship Table

| user1_id | user2_id |
|----------|----------|
| 1        | 2        |
| 2        | 4        |
| 2        | 5        |

---

## 🏗️ Schema Definition

```python
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

# Listens schema
listens_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("song_id", IntegerType(), False),
    StructField("day", DateType(), False)
])

# Friendship schema
friendship_schema = StructType([
    StructField("user1_id", IntegerType(), False),
    StructField("user2_id", IntegerType(), False)
])
```

---

## 📥 Data Preparation

```python
# Listens data
listens_data = [
    (1, 10, "2021-03-15"),
    (1, 11, "2021-03-15"),
    (1, 12, "2021-03-15"),
    (2, 10, "2021-03-15"),
    (2, 11, "2021-03-15"),
    (2, 12, "2021-03-15"),
    (3, 10, "2021-03-15"),
    (3, 11, "2021-03-15"),
    (3, 12, "2021-03-15"),
    (4, 10, "2021-03-15"),
    (4, 11, "2021-03-15"),
    (4, 13, "2021-03-15"),
    (5, 10, "2021-03-16"),
    (5, 11, "2021-03-16"),
    (5, 12, "2021-03-16")
]

# Friendship data
friendship_data = [
    (1, 2),
    (2, 4),
    (2, 5)
]
```

---

## 🗂️ Create DataFrames

```python
# Create Listens DataFrame
listens_df = spark.createDataFrame(listens_data, schema=listens_schema)
listens_df.show()

# Create Friendship DataFrame
friendship_df = spark.createDataFrame(friendship_data, schema=friendship_schema)
friendship_df.show()
```

---

## 👁️ Register as SQL Views

```python
listens_df.createOrReplaceTempView("Listens")
friendship_df.createOrReplaceTempView("Friendship")
```

---



In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, DateType

import datetime

listens_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("song_id", IntegerType(), False),
    StructField("day", DateType(), False)
])

listens_data = [
    (1, 10, datetime.date(2021, 3, 15)),
    (1, 11, datetime.date(2021, 3, 15)),
    (1, 12, datetime.date(2021, 3, 15)),
    (2, 10, datetime.date(2021, 3, 15)),
    (2, 11, datetime.date(2021, 3, 15)),
    (2, 12, datetime.date(2021, 3, 15)),
    (3, 10, datetime.date(2021, 3, 15)),
    (3, 11, datetime.date(2021, 3, 15)),
    (3, 12, datetime.date(2021, 3, 15)),
    (4, 10, datetime.date(2021, 3, 15)),
    (4, 11, datetime.date(2021, 3, 15)),
    (4, 13, datetime.date(2021, 3, 15)),
    (5, 10, datetime.date(2021, 3, 16)),
    (5, 11, datetime.date(2021, 3, 16)),
    (5, 12, datetime.date(2021, 3, 16))
]



# Friendship schema
friendship_schema = StructType([
    StructField("user1_id", IntegerType(), False),
    StructField("user2_id", IntegerType(), False)
])

# Friendship data
friendship_data = [
    (1, 2),
    (2, 4),
    (2, 5)
]
# Create Listens DataFrame
listens_df = spark.createDataFrame(listens_data, schema=listens_schema)


# Create Friendship DataFrame
friendship_df = spark.createDataFrame(friendship_data, schema=friendship_schema)


friendship_df.createOrReplaceTempView('Friendship')
listens_df.createOrReplaceTempView('Listens')
friendship_df.show()
listens_df.show()

In [0]:
%sql
with cte as (
    select user1_id  as self , user2_id  as friend from Friendship  
    union 
        select user2_id  as self , user1_id  as friend from Friendship  
)
,cte_2 as (
  select distinct  l1.user_id as self  , l2.user_id as friend  , 
  l1.song_id as song_id ,l1.day as day 
  from Listens l1 inner join Listens l2
  on l1.song_id  = l2.song_id 
  and l1.day         = l2.day        
  and l1.user_id  <> l2.user_id 

),cte3 as (
select  count(*)over(partition by cte_2.friend ,cte_2.self ,cte_2.day ) as cnt ,cte_2.* from cte_2
inner join cte on (cte.self = cte_2.self and cte.friend = cte_2.friend)
where cte_2.self < cte_2.friend
)
select distinct cte3.self as user1_id   , cte3.friend as user2_id  from cte3 where cnt >=3
